# SMARTS Overlay Explorer

This notebook combines one or more QM input files, rebuilds the OpenFF 2.3.0 comparisons, and overlays multiple SMARTS selections on the same QM reference distributions.

Use indexed SMARTS so only the requested geometry is measured:
- Bonds: provide SMARTS with exactly two mapped atoms (`:1`, `:2`).
- Angles: provide SMARTS with exactly three mapped atoms (`:1`, `:2`, `:3`), even if the SMARTS has additional unmapped atoms.

Edit `BOND_SMARTS_PATTERNS` and `ANGLE_SMARTS_PATTERNS` below. Edit `DATA_FILES_OVERRIDE` if you want to ignore automatic file discovery under `QM_ROOT`.

**FF-parameter filtering**: set `FF_ANGLE_PARAM_IDS` and/or `FF_BOND_PARAM_IDS` to restrict the data pool to only those geometry instances that were assigned a specific force-field parameter ID (e.g. `FF_ANGLE_PARAM_IDS = {'a10'}`). The SMARTS overlay patterns then split *within* that filtered subset.

In [13]:
from __future__ import annotations

import gzip
import hashlib
import pickle
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from rdkit import Chem

from mlip_optimizer import evaluate_against_qm
from mlip_optimizer.data import load_records
from mlip_optimizer.io import read_optimized_sdf


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise FileNotFoundError(f'Could not locate repo root from {start}')


REPO_ROOT = find_repo_root()
SCRIPT_DIR = REPO_ROOT / 'examples' / '07_single_model_benchmark'
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

from _shared import load_json_config, resolve_path

plt.style.use('seaborn-v0_8-whitegrid')

CONFIG_PATH = REPO_ROOT / 'examples' / '07_single_model_benchmark' / 'inputs' / 'compare_config.json'
POTENTIAL_NAME = 'openff-2.3.0'

# Default roots from your project layout.
QM_ROOT = REPO_ROOT / 'examples' / '01_download_datasets' / 'outputs' / 'optimization'
OPTIMIZED_ROOT = REPO_ROOT / 'examples' / '07_single_model_benchmark' / 'outputs' / 'optimized'

# Cache controls.
CACHE_ROOT = REPO_ROOT / 'examples' / '07_single_model_benchmark' / 'outputs' / 'cache' / 'qm_diffs'
USE_CACHE = True
FORCE_REBUILD_CACHE = False

# Optional overrides:
# - None: auto-discover all parquet/sdf QM files in QM_ROOT recursively
# - list[str|Path]: explicit QM files
DATA_FILES_OVERRIDE = None

# Indexed SMARTS requirements:
# - bond pattern must have exactly mapped atoms :1 and :2
# - angle pattern must have exactly mapped atoms :1, :2 and :3
BOND_SMARTS_PATTERNS = []

ANGLE_SMARTS_PATTERNS = [
    '[*:1]~[#6X3:2]~[*:3]',
    '[*:1]-[#6X3:2]=[*:3]',
    '[*:1]-[#6X3:2]-[*:3]',
    ]

# FF-parameter filtering (optional).
# Set FORCEFIELD_NAME and restrict data collection to specific parameter IDs.
# Examples:
#   FF_ANGLE_PARAM_IDS = {'a10'}           # only angles assigned to a10
#   FF_ANGLE_PARAM_IDS = {'a10', 'a11'}    # multiple params
#   FF_BOND_PARAM_IDS  = {'b15'}
# Leave as None to include all instances (no FF filtering).
FORCEFIELD_NAME = 'openff-2.3.0.offxml'
FF_BOND_PARAM_IDS: set[str] | None = None
FF_ANGLE_PARAM_IDS: set[str] | None = None

BOND_THRESHOLD = 0.1
ANGLE_THRESHOLD = 5.0
TORSION_THRESHOLD = 40.0

METRICS = ('bond', 'angle')

# a40 splits
```
ANGLE_SMARTS_PATTERNS = [
    '[*:1]~[#15:2]~[*:3]',
    '[#6:1]~[#15:2]~[#8:3]',
    '[#8:1]~[#15:2]~[#8:3]',
    '[#6:1]-[#15:2]=[#8:3]',
    '[#6:1]-[#15:2]-[#8:3]',
    '[#8:1]-[#15:2]=[#8:3]',
    '[#8:1]-[#15:2]-[#8:3]',
    '[*:1]-[#15:2]=[*:3]',
    '[*:1]-[#15:2]-[*:3]',
    ]


In [14]:
import importlib
import mlip_optimizer.analysis.bundle as _bundle_mod
import mlip_optimizer.analysis.smarts_overlay as _smarts_mod
importlib.reload(_bundle_mod)
importlib.reload(_smarts_mod)

from mlip_optimizer.analysis.bundle import (
    discover_qm_files,
    load_openff_bundle,
    load_bundles,
)

In [15]:
from mlip_optimizer.analysis.smarts_overlay import (
    normalize_patterns,
    compile_patterns,
    compile_indexed_patterns,
    collect_overlay_data,
    metric_label,
    metric_unit,
)
from mlip_optimizer.visualization.smarts_overlay import (
    plot_actual_overlay,
    plot_qm_split_overlay,
    plot_error_overlay,
)

In [ ]:
def run_overlay_analysis(
    bond_smarts_patterns=BOND_SMARTS_PATTERNS,
    angle_smarts_patterns=ANGLE_SMARTS_PATTERNS,
    data_files_override=DATA_FILES_OVERRIDE,
    forcefield_name=FORCEFIELD_NAME,
    ff_bond_param_ids=FF_BOND_PARAM_IDS,
    ff_angle_param_ids=FF_ANGLE_PARAM_IDS,
):
    config, bundles = load_bundles(
        QM_ROOT,
        OPTIMIZED_ROOT,
        POTENTIAL_NAME,
        data_files_override=data_files_override,
        cache_root=CACHE_ROOT,
        use_cache=USE_CACHE,
        force_rebuild_cache=FORCE_REBUILD_CACHE,
        bond_threshold=BOND_THRESHOLD,
        angle_threshold=ANGLE_THRESHOLD,
        torsion_threshold=TORSION_THRESHOLD,
        label_forcefield_name=forcefield_name,
    )
    analysis, summary = collect_overlay_data(
        bundles,
        bond_smarts_patterns,
        angle_smarts_patterns,
        POTENTIAL_NAME,
        metrics=METRICS,
        forcefield_name=forcefield_name,
        ff_bond_param_ids=ff_bond_param_ids,
        ff_angle_param_ids=ff_angle_param_ids,
    )

    display(Markdown('## Input summary'))
    input_rows = [
        {
            'dataset': bundle['dataset_name'],
            'records': len(bundle['records']),
            'optimized_sdf': str(bundle['optimized_sdf']),
        }
        for bundle in bundles
    ]
    display(pd.DataFrame(input_rows))

    # Show active FF-param filters if any were set
    if ff_bond_param_ids is not None or ff_angle_param_ids is not None:
        filter_rows = []
        if ff_angle_param_ids is not None:
            filter_rows.append({'metric': 'angle', 'ff_param_ids': ', '.join(sorted(ff_angle_param_ids))})
        if ff_bond_param_ids is not None:
            filter_rows.append({'metric': 'bond', 'ff_param_ids': ', '.join(sorted(ff_bond_param_ids))})
        display(Markdown('## FF-parameter filter active'))
        display(pd.DataFrame(filter_rows))

    display(Markdown('## SMARTS summary'))
    display(summary)

    for metric in METRICS:
        display(Markdown(f'### {metric_label(metric)}'))
        plot_actual_overlay(analysis, metric)
        plot_qm_split_overlay(analysis, metric)
        plot_error_overlay(analysis, metric)

    return config, bundles, analysis, summary


config, bundles, analysis, summary = run_overlay_analysis()

2026-06-05 08:36:53,011 - mlip_optimizer.data.readers - INFO - Reading parquet: /home/maverick/Desktop/CNT-work/EC_Project/NSP_datasets/mlip_optimizer/examples/01_download_datasets/outputs/optimization/OpenFF_NSP_Optimization_Set_1_Nitrogen_v4_0_20260306T142716/OpenFF_NSP_Optimization_Set_1_Nitrogen_v4_0.parquet


2026-06-05 08:36:56,153 - mlip_optimizer.data.grouping - INFO - Grouped 2976 rows into 528 molecules (2976 total conformers)


cache hit  -> loaded: /home/maverick/Desktop/CNT-work/EC_Project/NSP_datasets/mlip_optimizer/examples/07_single_model_benchmark/outputs/cache/qm_diffs/OpenFF_NSP_Optimization_Set_1_Nitrogen_v4_0__openff-2_3_0__71e0c912541b.pkl.gz


2026-06-05 08:38:09,012 - mlip_optimizer.data.readers - INFO - Reading parquet: /home/maverick/Desktop/CNT-work/EC_Project/NSP_datasets/mlip_optimizer/examples/01_download_datasets/outputs/optimization/OpenFF_NSP_Optimization_Set_1_Phosphorus_v4_0_20260306T142948/OpenFF_NSP_Optimization_Set_1_Phosphorus_v4_0.parquet


ff cache miss -> wrote: /home/maverick/Desktop/CNT-work/EC_Project/NSP_datasets/mlip_optimizer/examples/07_single_model_benchmark/outputs/cache/qm_diffs/OpenFF_NSP_Optimization_Set_1_Nitrogen_v4_0__ff_openff-2_3_0_offxml__27efe5515b9f.pkl.gz


...

cache hit  -> loaded: /home/maverick/Desktop/CNT-work/EC_Project/NSP_datasets/mlip_optimizer/examples/07_single_model_benchmark/outputs/cache/qm_diffs/OpenFF_NSP_Optimization_Set_1_Phosphorus_v4_0__openff-2_3_0__d40d829ddb6a.pkl.gz


2026-06-05 08:39:04,482 - mlip_optimizer.data.readers - INFO - Reading parquet: /home/maverick/Desktop/CNT-work/EC_Project/NSP_datasets/mlip_optimizer/examples/01_download_datasets/outputs/optimization/OpenFF_NSP_Optimization_Set_1_Sulfur_v4_0_20260306T142817/OpenFF_NSP_Optimization_Set_1_Sulfur_v4_0.parquet


ff cache miss -> wrote: /home/maverick/Desktop/CNT-work/EC_Project/NSP_datasets/mlip_optimizer/examples/07_single_model_benchmark/outputs/cache/qm_diffs/OpenFF_NSP_Optimization_Set_1_Phosphorus_v4_0__ff_openff-2_3_0_offxml__afd9b45f41d7.pkl.gz


2026-06-05 08:39:08,067 - mlip_optimizer.data.grouping - INFO - Grouped 3570 rows into 588 molecules (3570 total conformers)
2026-06-05 08:39:16,179 - mlip_optimizer.io - WARNING - No optimized data for molecule 295 in optimized_openff-2_3_0_20260326T103257.sdf (Opt. fail)
2026-06-05 08:39:16,301 - mlip_optimizer.io - WARNING - No optimized data for molecule 367 in optimized_openff-2_3_0_20260326T103257.sdf (Opt. fail)
2026-06-05 08:39:16,316 - mlip_optimizer.io - WARNING - No optimized data for molecule 375 in optimized_openff-2_3_0_20260326T103257.sdf (Opt. fail)


cache hit  -> loaded: /home/maverick/Desktop/CNT-work/EC_Project/NSP_datasets/mlip_optimizer/examples/07_single_model_benchmark/outputs/cache/qm_diffs/OpenFF_NSP_Optimization_Set_1_Sulfur_v4_0__openff-2_3_0__399e8c6fddb7.pkl.gz


### BESMARTS split inputs

The call below groups all matching angle examples for the parent SMARTS pattern into a few value-based clusters, then tries to describe each cluster with a child SMARTS.

- `PARENT_ANGLE_SMARTS`: the parent pattern to split. Here it is the phosphorus angle `[*:1]~[#15:2]~[*:3]`.
- `BESMARTS_MAX_K`: the maximum number of value clusters to try. Higher values can produce more, smaller groups.
- `BESMARTS_MIN_CLUSTER_SIZE`: the smallest number of angle examples allowed in any cluster. This keeps the split from overfitting tiny groups.
- `BESMARTS_DEPTH`: how much chemical neighborhood context to include when building each child SMARTS. `1` uses only the matched angle atoms and their immediate pattern context; `2` adds more neighboring chemistry and can produce more specific SMARTS.
- `recommended_smarts`: the output child SMARTS patterns that are ready to feed back into the overlay plots.

In [ ]:
import importlib
import besmarts_angle_split as _besmarts_angle_split_mod
importlib.reload(_besmarts_angle_split_mod)
from besmarts_angle_split import propose_child_angle_smarts

PARENT_ANGLE_SMARTS = '[*:1]~[#15:2]~[*:3]'
BESMARTS_MAX_K = 6  # try 10 for more clusters; 0 means no upper limit on k
BESMARTS_MIN_CLUSTER_SIZE = 80
BESMARTS_DEPTH = 6  # neighbourhood depth for fragment intersection; try 2 for richer SMARTS

split_result = propose_child_angle_smarts(
    bundles,
    PARENT_ANGLE_SMARTS,
    max_k=BESMARTS_MAX_K,
    min_cluster_size=BESMARTS_MIN_CLUSTER_SIZE,
    depth=BESMARTS_DEPTH,
)

parent_df = split_result['parent_df']
partition = split_result['partition']
child_df = split_result['child_df'].copy()
recommended_smarts = split_result['recommended_smarts']

parent_summary = pd.DataFrame(
    [
        {'metric': 'parent_smarts', 'value': PARENT_ANGLE_SMARTS},
        {'metric': 'n_entries', 'value': len(split_result['entries'])},
        {'metric': 'n_unique_molecules', 'value': parent_df['molecule'].nunique()},
        {'metric': 'chosen_k', 'value': partition['k']},
        {'metric': 'rss', 'value': partition['rss']},
        {'metric': 'bic', 'value': partition['bic']},
    ]
)

display(Markdown('## BESMARTS parent-angle split summary'))
display(parent_summary)

display(Markdown('### Partition cluster statistics'))
display(partition['clusters'])

display(Markdown('### Derived child SMARTS'))
display(child_df)

if recommended_smarts:
    display(Markdown('### Recommended child SMARTS for overlay'))
    display(pd.DataFrame({'angle_smarts': recommended_smarts}))
else:
    display(Markdown(
        '**No explicit child SMARTS were produced.** '
        'All clusters yielded `None` from BESMARTS intersection — '
        'try increasing `BESMARTS_DEPTH` to 2.'
    ))

RUN_CHILD_OVERLAY = bool(recommended_smarts)
if RUN_CHILD_OVERLAY:
    _, _, child_analysis, child_summary = run_overlay_analysis(
        bond_smarts_patterns=[],
        angle_smarts_patterns=recommended_smarts,
        data_files_override=config.get('data_files', None),
    )
    display(Markdown('### Child SMARTS overlay summary'))
    display(child_summary)
